In [ ]:
!nvidia-smi


Fri Nov 14 09:52:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install ultralytics==8.3.5 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.8/882.8 kB 49.0 MB/s eta 0:00:00


In [ ]:
import os
os.environ["WANDB_MODE"] = "offline"
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
!pip install roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 66.0 MB/s eta 0:00:00


In [ ]:
version_of_dataset = 9
from roboflow import Roboflow
rf = Roboflow(api_key="Lke7S18T11cvHXQiSlnE")
project = rf.workspace("vest-neclx").project("common_v1-ddm81")
dataset = project.version(version_of_dataset).download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Common_V1-9 in yolov11:: 100%|██████████| 1907/1907 [00:00<00:00, 4019.37it/s]


In [ ]:

base_path = "/content/Common_V1-9"
for folder in ["train", "valid"]:
    path = os.path.join(base_path, folder, "images")
    count = len(os.listdir(path))
    print(f"{folder}: {count} изображений")

train: 814 изображений
valid: 135 изображений


In [ ]:
model = YOLO("yolo11n.pt")

100%|██████████| 5.35M/5.35M [00:00<00:00, 160MB/s]


# Версия 1


In [ ]:
os.environ["WANDB_MODE"] = "offline"

model2 = YOLO("yolo11m.pt")


results = model2.train(
    data="/content/Common_V1-7/data.yaml",
    epochs=150,
    patience=20,
    imgsz=640,
    optimizer='AdamW',                       # AdamW лучше для маленьких объектов
    lr0=0.001,
    weight_decay=0.0005,
    warmup_momentum=0.8,
    warmup_epochs=3,
    pretrained=True,
    batch=16,
    device=0,
    name="ppe_yolo11m_optimized",
    workers=2,
    box=7.5,     # повысим точность по координатам
    cls=0.5,     # важна правильная классификация (helmet vs head)
    dfl=1.5,     # стандартно
    cos_lr=True
)

New https://pypi.org/project/ultralytics/8.3.223 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11m.pt, data=/content/Common_V1-7/data.yaml, epochs=150, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=None, name=ppe_yolo11m_optimized4, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=F

Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLO11n...
AMP: checks passed ✅


train: Scanning /content/Common_V1-7/train/labels.cache... 545 images, 0 backgrounds, 0 corrupt: 100%|██████████| 545/545 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


Argument(s) 'quality_lower' are not valid for transform ImageCompression
val: Scanning /content/Common_V1-7/valid/labels.cache... 139 images, 1 backgrounds, 0 corrupt: 100%|██████████| 139/139 [00:00<?, ?it/s]


...

    147/150      8.66G     0.5798     0.3428     0.9702          9        640: 100%|██████████| 35/35 [00:19<00:00,  1.81it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  2.39it/s]


                   all        139       1239      0.849      0.733      0.795      0.501
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 127, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

147 epochs completed in 0.981 hours.
Optimizer stripped from runs/detect/ppe_yolo11m_optimized4/weights/last.pt, 40.5MB
Optimizer stripped from runs/detect/ppe_yolo11m_optimized4/weights/best.pt, 40.5MB

Validating runs/detect/ppe_yolo11m_optimized4/weights/best.pt...
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11m summary (fused): 303 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:05<00:00,  1.05s/it]


                   all        139       1239      0.847      0.732      0.806      0.508
                  body         75        137      0.791      0.664      0.752      0.385
                 glove         61        139      0.746      0.486      0.601      0.279
                  head         33         65      0.902      0.631      0.757      0.485
                helmet        113        238      0.932       0.92       0.95       0.67
                 human        137        330      0.906       0.88      0.913       0.69
                  palm         72        155      0.778      0.632      0.749      0.382
                  vest         88        175      0.875      0.909      0.923      0.667
Speed: 0.3ms preprocess, 11.3ms inference, 0.0ms loss, 6.6ms postprocess per image
Results saved to runs/detect/ppe_yolo11m_optimized4


lr/pg0,█████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
lr/pg1,███████▇▇▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
lr/pg2,▆███████▇▇▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
metrics/mAP50(B),▁▂▃▄▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇█████████████████████
metrics/mAP50-95(B),▁▃▃▃▄▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████████████████
metrics/precision(B),▂▁▂▂▄▅▆▅▆▆▆▆▇▆▇▇▇▇▇▇▇▇█▇███▇██▇▇██████▇█
metrics/recall(B),▁▄▄▄▄▅▆▆▆▆▆▆▇▇▇▇███▇▇▇█▇▇██▇▇▇██▇███████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


# Версия 2

In [ ]:
os.environ["WANDB_MODE"] = "offline"

model2 = YOLO("yolo11m.pt")



In [ ]:
results2 = model2.train(
    data="/content/Common_V1-7/data.yaml",
    epochs=150,
    patience=15,
    imgsz=768,
    batch=8,
    device=0,
    name="ppe_yolo11m_optimized",
    workers=2,
    augment=True,
    auto_augment="light",
    mosaic=0.5,
    mixup=0.3,
    cos_lr=True,
    val=True
)

New https://pypi.org/project/ultralytics/8.3.223 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11m.pt, data=/content/Common_V1-7/data.yaml, epochs=150, time=None, patience=15, batch=8, imgsz=768, save=True, save_period=-1, cache=False, device=0, workers=2, project=None, name=ppe_yolo11m_optimized5, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fal

Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLO11n...
AMP: checks passed ✅


train: Scanning /content/Common_V1-7/train/labels.cache... 545 images, 0 backgrounds, 0 corrupt: 100%|██████████| 545/545 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



Argument(s) 'quality_lower' are not valid for transform ImageCompression
val: Scanning /content/Common_V1-7/valid/labels.cache... 139 images, 1 backgrounds, 0 corrupt: 100%|██████████| 139/139 [00:00<?, ?it/s]


...

                   all        139       1239      0.836      0.722      0.797      0.495
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 51, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



66 epochs completed in 0.683 hours.
Optimizer stripped from runs/detect/ppe_yolo11m_optimized5/weights/last.pt, 40.5MB
Optimizer stripped from runs/detect/ppe_yolo11m_optimized5/weights/best.pt, 40.5MB

Validating runs/detect/ppe_yolo11m_optimized5/weights/best.pt...
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11m summary (fused): 303 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:14<00:00,  1.63s/it]


                   all        139       1239      0.826      0.719      0.807      0.516
                  body         75        137      0.788      0.642      0.748        0.4
                 glove         61        139      0.687      0.489      0.618      0.307
                  head         33         65      0.843      0.631      0.759      0.506
                helmet        113        238      0.933       0.93      0.947      0.656
                 human        137        330      0.876      0.852       0.91      0.693
                  palm         72        155      0.816      0.581      0.746      0.384
                  vest         88        175      0.837      0.912      0.917      0.668
Speed: 0.3ms preprocess, 87.0ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/ppe_yolo11m_optimized5


lr/pg0,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr/pg1,██████████▇▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▁
lr/pg2,██████████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁▁
metrics/mAP50(B),▁▃▂▃▅▄▄▅▅▅▆▆▆▅▅▆▆▇▆▅▆▅▆▆▆▆▆▇▆██▇▆▇▆▆▆▆▅█
metrics/mAP50-95(B),▁▄▄▄▄▅▅▄▅▅▆▅▅▆▆▆▆▆▆▆▆▆▆▅▇▆▆▇▇▇▆▆▆▇▇▇▆▆▆█
metrics/precision(B),▁▄▅▃▄▅▄▆█▄▅▅▅▆▆▇▆▅▆▇▅▆▆▆▇▅▅▇▆▆▇▇█▆▆▆▆▇▅▇
metrics/recall(B),▃▁▃▃▄▄▂▅▃▆▃▄▆▆▅▆▇▇▆▆▇█▆▇▇▅▇▅▇▆▆▆▅▂▇█▇█▆▇
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


In [ ]:
def save_model(model: YOLO, path: str):

    model.export(format="pt")  # сохраняет в формате .pt
    os.rename("yolo11n.pt", path)  # переименовываем в указанный путь
    print(f"Модель сохранена по пути: {path}")

def load_model(path: str, device: int = 0) -> YOLO:

    model = YOLO(path)
    model.to(device)
    print(f"Модель загружена с {path} на устройство {device}")
    return model

# Версия 3

In [ ]:
model3 = YOLO("yolo11m.pt")

data_yaml = "/content/Common_V1-7/data.yaml"

results3 = model3.train(
    data="/content/Common_V1-7/data.yaml",
    epochs=150,
    patience=15,
    imgsz=[512, 768, 1024],
    batch=8,
    device=0,
    name="ppe_yolo11m_optimized",
    workers=2,
    augment=True,
    auto_augment="light",
    mosaic=0.5,
    mixup=0.3,
    cos_lr=True,
    val=True,
    rect=False,
    cache=False,
    classes=[1, 5]
)

100%|██████████| 38.8M/38.8M [00:00<00:00, 192MB/s]


New https://pypi.org/project/ultralytics/8.3.225 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11m.pt, data=/content/Common_V1-7/data.yaml, epochs=150, time=None, patience=15, batch=8, imgsz=[512, 768, 1024], save=True, save_period=-1, cache=False, device=0, workers=2, project=None, name=ppe_yolo11m_optimized, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=[1, 5], retina_masks=False, embed=N

100%|██████████| 755k/755k [00:00<00:00, 35.7MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  7                  -1  1   2360320  ultralytics

invalid escape sequence '\/'


...

                   all        139        294      0.731      0.592      0.671      0.323
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 100, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



115 epochs completed in 1.854 hours.
Optimizer stripped from runs/detect/ppe_yolo11m_optimized/weights/last.pt, 40.6MB
Optimizer stripped from runs/detect/ppe_yolo11m_optimized/weights/best.pt, 40.6MB

Validating runs/detect/ppe_yolo11m_optimized/weights/best.pt...
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11m summary (fused): 303 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:20<00:00,  2.27s/it]


                   all        139        294      0.782      0.573      0.678      0.344
                 glove         61        139      0.703      0.504      0.603      0.299
                  palm         72        155      0.862      0.643      0.754       0.39
Speed: 0.6ms preprocess, 139.5ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs/detect/ppe_yolo11m_optimized


lr/pg0,▃█████████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▂▂▂▂▂▁▁▁▁▁
lr/pg1,▃█████▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▁▁▁▁
lr/pg2,█████████▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁
metrics/mAP50(B),▃▂▁▁▂▃▃▅▄▄▆▆▆▆▆▇▇▇▆▇▇▇▇▇▇▇▇▇█▇█▇████████
metrics/mAP50-95(B),▃▁▂▂▃▄▅▅▅▆▆▆▆▆▅▆▆▅▇▇▆▆▇▇▆▇▇▇▇▇▇▇███▇█▇▇█
metrics/precision(B),▃▂▁▁▂▁▃▃▄▅▄▂▄▆▅▆▅▆▇▅▆▇▆▆▇▇▇▆█▆▆▆██▇▆▇▆▇█
metrics/recall(B),▃▁▁▂▄▃▄▄▅▅▆▅▆▆▅▅▆▆▇▆▅▆▆▇▆▇▆▇█▆▇▆▇█▇█▇█▇▇
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


# Версия 4

In [ ]:
model4 = YOLO("yolo11m.pt")
results = model4.train(
    data="/content/Common_V1-8/data.yaml",
    epochs=250,
    patience=35,
    imgsz=945,
    optimizer='AdamW',                       # AdamW лучше для маленьких объектов
    weight_decay=0.0005,
    warmup_momentum=0.8,
    warmup_epochs=3,
    pretrained=True,
    batch=8,
    rect=False,
    device=0,
    name="ppe_yolo11m_optimized",
    workers=3,
    box=7.5,     # повысим точность по координатам
    cls=0.5,     # важна правильная классификация (helmet vs head)
    dfl=1.5,     # стандартно
    cos_lr=True,
    augment=True,                  # включить базовые аугментации (flip, HSV)
    auto_augment="light",          # лёгкая автоматическая аугментация
    mosaic=0.5,                     # Mosaic для маленьких объектов
    mixup=0.3,                      # MixUp для улучшения классификации
    flipud=0.0,                     # вертикальные flip не используем
    fliplr=0.5,                     # горизонтальные flip
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4
)

100%|██████████| 38.8M/38.8M [00:00<00:00, 204MB/s]


New https://pypi.org/project/ultralytics/8.3.225 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11m.pt, data=/content/Common_V1-8/data.yaml, epochs=250, time=None, patience=35, batch=8, imgsz=945, save=True, save_period=-1, cache=False, device=0, workers=3, project=None, name=ppe_yolo11m_optimized, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fals

100%|██████████| 755k/755k [00:00<00:00, 23.0MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  7                  -1  1   2360320  ultralytics

invalid escape sequence '\/'


...

    163/250      9.91G      1.198      1.018      1.496          3        960: 100%|██████████| 88/88 [00:56<00:00,  1.56it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.07it/s]

                   all        127       1128      0.763      0.739      0.772       0.47



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    164/250      9.77G      1.182      1.001      1.485        133        960:  91%|█████████ | 80/88 [00:51<00:05,  1.54it/s]


KeyboardInterrupt: 

In [ ]:
model4.save("/content/ppe_best_4.pt")

In [ ]:
!zip -r /content/all_content2.zip /content/runs
from google.colab import files
files.download('/content/all_content2.zip')

  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/ppe_yolo11m_optimized/ (stored 0%)
  adding: content/runs/detect/ppe_yolo11m_optimized/weights/ (stored 0%)
  adding: content/runs/detect/ppe_yolo11m_optimized/weights/best.pt (deflated 39%)
  adding: content/runs/detect/ppe_yolo11m_optimized/weights/last.pt (deflated 39%)
  adding: content/runs/detect/ppe_yolo11m_optimized/train_batch1.jpg (deflated 6%)
  adding: content/runs/detect/ppe_yolo11m_optimized/train_batch2.jpg (deflated 3%)
  adding: content/runs/detect/ppe_yolo11m_optimized/events.out.tfevents.1762530037.f2f2e1219784.1937.0 (deflated 88%)
  adding: content/runs/detect/ppe_yolo11m_optimized/labels.jpg (deflated 23%)
  adding: content/runs/detect/ppe_yolo11m_optimized/events.out.tfevents.1762541241.f2f2e1219784.1937.4 (deflated 94%)
  adding: content/runs/detect/ppe_yolo11m_optimized/train_batch0.jpg (deflated 7%)
  adding: content/runs/detect/ppe_yolo11m_optimized/re

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r /content/all_content.zip /content
from google.colab import files
files.download('/content/all_content.zip')

In [ ]:
model5 = YOLO("yolo11n.pt")
results = model5.train(
    data="/content/Common_V1-8/data.yaml",
    epochs=250,
    patience=35,
    imgsz=945,
    optimizer='AdamW',
    weight_decay=0.0005,
    warmup_momentum=0.8,
    warmup_epochs=3,
    pretrained=True,
    batch=8,
    rect=False,
    device=0,
    name="ppe_yolo11m_optimized",
    workers=3,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    cos_lr=True,
    augment=True,
    auto_augment="light",
    mosaic=0.5,
    mixup=0.3,
    flipud=0.0,
    fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4
)

100%|██████████| 5.35M/5.35M [00:00<00:00, 101MB/s]


New https://pypi.org/project/ultralytics/8.3.226 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/Common_V1-8/data.yaml, epochs=250, time=None, patience=35, batch=8, imgsz=945, save=True, save_period=-1, cache=False, device=0, workers=3, project=None, name=ppe_yolo11m_optimized, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fals

100%|██████████| 755k/755k [00:00<00:00, 23.9MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

invalid escape sequence '\/'


...

                   all        127       1128      0.784        0.7      0.774      0.481
EarlyStopping: Training stopped early as no improvement observed in last 35 epochs. Best results observed at epoch 178, best model saved as best.pt.
To update EarlyStopping(patience=35) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



213 epochs completed in 1.980 hours.
Optimizer stripped from runs/detect/ppe_yolo11m_optimized/weights/last.pt, 5.6MB
Optimizer stripped from runs/detect/ppe_yolo11m_optimized/weights/best.pt, 5.6MB

Validating runs/detect/ppe_yolo11m_optimized/weights/best.pt...
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 238 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:17<00:00,  2.19s/it]


                   all        127       1128      0.799       0.71      0.785      0.497
                  body         68        129       0.76      0.662      0.732      0.392
                 glove         54        126      0.701      0.381      0.544      0.255
                  head         31         62      0.771      0.651      0.733       0.47
                helmet        102        213      0.897       0.94      0.946      0.673
                  palm         64        133      0.743      0.632      0.712      0.368
                person        125        305      0.897      0.833      0.911      0.649
                  vest         81        160      0.822      0.869      0.919      0.674
Speed: 0.5ms preprocess, 115.1ms inference, 0.0ms loss, 4.4ms postprocess per image
Results saved to runs/detect/ppe_yolo11m_optimized


lr/pg0,████████▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁
lr/pg1,███████▇▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁
lr/pg2,█████▇▇▇▇▇▆▆▆▆▆▆▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
metrics/mAP50(B),▁▃▄▃▃▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████████
metrics/mAP50-95(B),▁▂▂▂▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇█████████████
metrics/precision(B),▅▁▃▅▆▆▆▆▆▆▆▇▇▇▇▇▇▆▇▆▆▇▇▇▇██▇▇▇██████████
metrics/recall(B),▁▃▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇█▇█▇█▇██████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


In [ ]:
model5_1 = YOLO("runs/detect/ppe_yolo11m_optimized/weights/last.pt")
results_continue = model5_1.train(
    data="/content/Common_V1-8/data.yaml",
    epochs=500,
    batch=8,
    imgsz=945,
    patience=35,
    pretrained=True,       # продолжить с последнего состояния
    name="ppe_yolo11m_optimized",
    optimizer='AdamW',
    weight_decay=0.0005,
    warmup_momentum=0.8,
    resume="True",
    warmup_epochs=3,
    rect=False,
    device=0,
    workers=3,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    cos_lr=True,
    augment=True,
    auto_augment="light",
    mosaic=0.5,
    mixup=0.3,
    flipud=0.0,
    fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4
)

# Версия 6

In [ ]:
model6 = YOLO("yolo11n.pt")
results = model6.train(
    data="/content/Common_V1-9/data.yaml",
    epochs=350,
    patience=45,
    imgsz=960,
    optimizer='AdamW',
    weight_decay=0.0005,
    warmup_momentum=0.8,
    warmup_epochs=3,
    pretrained=True,
    batch=8,
    rect=False,
    device=0,
    name="ppe_yolo11nV5",
    box=9.0,
    cls=1.0,
    dfl=1.8,
    kobj=1.4,
    cos_lr=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4
)

New https://pypi.org/project/ultralytics/8.3.228 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/Common_V1-9/data.yaml, epochs=350, time=None, patience=45, batch=8, imgsz=960, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=ppe_yolo11nV5, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save

100%|██████████| 755k/755k [00:00<00:00, 65.9MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

invalid escape sequence '\/'


Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLO11n...
AMP: checks passed ✅


...

                   all        135       1159      0.834       0.75      0.807      0.507


EarlyStopping: Training stopped early as no improvement observed in last 45 epochs. Best results observed at epoch 237, best model saved as best.pt.
To update EarlyStopping(patience=45) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

282 epochs completed in 2.618 hours.
Optimizer stripped from runs/detect/ppe_yolo11nV5/weights/last.pt, 5.6MB
Optimizer stripped from runs/detect/ppe_yolo11nV5/weights/best.pt, 5.6MB

Validating runs/detect/ppe_yolo11nV5/weights/best.pt...
Ultralytics 8.3.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 238 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:04<00:00,  1.97it/s]


                   all        135       1159      0.861      0.756      0.814      0.513
                  body         70        126      0.791      0.662      0.756      0.381
                 glove         58        132      0.821       0.52      0.625      0.306
                  head         31         61      0.907      0.689      0.762      0.501
                helmet        112        220      0.941      0.955      0.967      0.702
                  palm         68        147       0.78      0.698       0.75      0.388
                person        134        307      0.899      0.863      0.906      0.631
                  vest         88        166      0.888      0.903      0.931      0.682
Speed: 0.7ms preprocess, 7.9ms inference, 0.0ms loss, 7.3ms postprocess per image
Results saved to runs/detect/ppe_yolo11nV5


lr/pg0,█▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
lr/pg1,▃▅█████████▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▂▂▂▂▂▂▂▁▁▁
lr/pg2,▅████████▇▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▁▁▁▁▁
metrics/mAP50(B),▁▄▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇██████████████████
metrics/mAP50-95(B),▁▅▅▆▆▆▆▆▇▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████████████
metrics/precision(B),▁▆▆▆▇▆▇▇▇▇▆▇▇▇▇▇█████▇██████████████████
metrics/recall(B),▂▁▄▄▅▅▅▆▅▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇█▇████████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


In [ ]:
!zip -r V6.zip /content/runs/detect

  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/ppe_yolo11nV5/ (stored 0%)
  adding: content/runs/detect/ppe_yolo11nV5/val_batch0_pred.jpg (deflated 4%)
  adding: content/runs/detect/ppe_yolo11nV5/val_batch0_labels.jpg (deflated 4%)
  adding: content/runs/detect/ppe_yolo11nV5/results.csv (deflated 86%)
  adding: content/runs/detect/ppe_yolo11nV5/F1_curve.png (deflated 6%)
  adding: content/runs/detect/ppe_yolo11nV5/args.yaml (deflated 52%)
  adding: content/runs/detect/ppe_yolo11nV5/events.out.tfevents.1763114499.fb91ebc9e144.952.0 (deflated 84%)
  adding: content/runs/detect/ppe_yolo11nV5/train_batch0.jpg (deflated 3%)
  adding: content/runs/detect/ppe_yolo11nV5/PR_curve.png (deflated 9%)
  adding: content/runs/detect/ppe_yolo11nV5/val_batch1_pred.jpg (deflated 5%)
  adding: content/runs/detect/ppe_yolo11nV5/val_batch2_pred.jpg (deflated 4%)
  adding: content/runs/detect/ppe_yolo11nV5/val_batch2_labels.jpg (deflated 4%)
  adding: content/runs/detect/ppe_yolo11

In [ ]:
from google.colab import files
files.download("V6.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>